# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [53]:
load_dotenv(override=True)
google_api_key = os.getenv('GEMINI_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:7]}")
else:
    print("Ollama API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

Ollama API Key exists and begins ollama
Google API Key exists and begins AI
Grok API Key exists and begins xai-


In [54]:
# Connect to client libraries
gemini_url = os.getenv('GEMINI_BASE_URL')
grok_url = os.getenv('GROK_BASE_URL')
ollama_url = os.getenv('OLLAMA_BASE_URL')

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# List Gemini models
try:
    gemini_models = gemini.models.list()
    all_gemini = [m.id for m in gemini_models.data]
    print("Gemini Models:", all_gemini)
except Exception as e:
    print("Error fetching Gemini models:", str(e))

# List Grok models
try:
    grok_models = grok.models.list()
    all_grok = [m.id for m in grok_models.data]
    print("Grok Models:", all_grok)
except Exception as e:
    print("Error fetching Grok models:", str(e))

# List Ollama models
try:
    ollama_models = ollama.models.list()
    all_ollama = [m.id for m in ollama_models.data]
    print("Ollama Models:", all_ollama)
except Exception as e:
    print("Error fetching Ollama models:", str(e))

Gemini Models: ['models/embedding-gecko-001', 'models/gemini-2.5-pro-preview-03-25', 'models/gemini-2.5-flash', 'models/gemini-2.5-pro-preview-05-06', 'models/gemini-2.5-pro-preview-06-05', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash-exp', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-exp-image-generation', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.0-flash-lite-preview-02-05', 'models/gemini-2.0-flash-lite-preview', 'models/gemini-2.0-pro-exp', 'models/gemini-2.0-pro-exp-02-05', 'models/gemini-exp-1206', 'models/gemini-2.0-flash-thinking-exp-01-21', 'models/gemini-2.0-flash-thinking-exp', 'models/gemini-2.0-flash-thinking-exp-1219', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/learnlm-2.0-flash-experimental', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'model

In [55]:
models = [ 
  "gemini-2.5-pro", 
  "gemini-2.5-flash", 
  "grok-4-1-fast-reasoning", 
  "grok-4-1-fast-non-reasoning", 
  "grok-4", 
  "kimi-k2-thinking:cloud",
  "qwen3-coder:480b-cloud",
  "gpt-oss:120b-cloud",
  "gemini-3-pro-preview:latest"
]

clients = { 
  "gemini-2.5-pro": gemini, 
  "gemini-2.5-flash": gemini, 
  "grok-4-1-fast-reasoning": grok, 
  "grok-4-1-fast-non-reasoning": grok, 
  "grok-4": grok, 
  "kimi-k2-thinking:cloud": ollama, 
  "qwen3-coder:480b-cloud": ollama,
  "gpt-oss:120b-cloud": ollama,
  "gemini-3-pro-preview:latest": ollama,
}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [56]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': '/home/ubuntu/.cargo/bin/rustc',
  'version': 'rustc 1.91.1 (ed61e7d7e 2025-11-07)',
  'host_triple': 'x86_64-unknown-linux-gnu',
  'release': '1.91.1',
  'commit_hash': 'ed61e7d7e242494fb7057f2657300d9e77bb4fcb'},
 'cargo': {'path': '/home/ubuntu/.cargo/bin/cargo',
  'version': 'cargo 1.91.1 (ea2d97820 2025-10-10)'},
 'rustup': {'path': '/home/ubuntu/.cargo/bin/rustup',
  'version': 'rustup 1.28.2 (e4f3ad6f8 2025-04-28)',
  'active_toolchain': 'stable-x86_64-unknown-linux-gnu (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-unknown-linux-gnu (active, default)'],
  'targets_installed': ['x86_64-unknown-linux-gnu']},
 'rust_analyzer': {'path': '/home/ubuntu/.cargo/bin/rust-analyzer'},
 'env': {'CARGO_HOME': '/home/ubuntu/.cargo',
  'RUSTUP_HOME': '/home/ubuntu/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"/home/ubuntu/.cargo/bin/cargo" build',
  '"/home/ubuntu/.cargo/bin/cargo" run',
  '"/

In [57]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

you can check rustc --version and gcc --version to see if I have rustc and gcc installed.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = grok.chat.completions.create(model="grok-4-1-fast-reasoning", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You already have a Rust toolchain installed (`rustc 1.91.1`), so no installation is needed.

Use these commands for maximum runtime performance (release optimizations, native CPU features, LTO, and minimal codegen units; compile time will be slower as a tradeoff):

```python
import subprocess

compile_command = [
    "rustc",
    "main.rs",
    "-O",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-o", "main"
]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [58]:
compile_command = [
    rust_info["rustc"]["path"],
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-o", "main",
]

run_command = ["./main"]

## And now, on with the main task

In [59]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [60]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [61]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [62]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [63]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [64]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [65]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [66]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7864/: Operation not supported


## RESULTS!
Python: 53.814 seconds
  
Gemini 2.5 Pro: Worked but it didn't put timing, manual fix .000614
Gemini 2.5 Flash: .723 
Grok 4: 0.000641  
Grok 4-1 Fast Reasoning: .000354 seconds  
Grok 4-1 Fast Non-Reasoning: FAIL  
Ollama Kimi K2 Thinking:cloud: .000389 
Ollama gemini-3-pro-preview:latest: FAIL remove the f in the format string: .000383
Ollama gpt-oss-120b-cloud: .000350
Ollama qwen3-coder:480b-cloud: FAIL

**1st place: gpt-oss-120b-cloud: 0.000350**  
**2nd place: Ollama Kimi K2 Thinking:cloud: .000392**
**3rd place: Gemini 3 Pro: .000383**


In [67]:
print(f"In my experimenet, the GROK 4-1 Fast Reasoning model outcome is {51.57/0.000354:,.0f} times faster than the Python code.")

In my experimenet, the GROK 4-1 Fast Reasoning model outcome is 145,678 times faster than the Python code.
